In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
import pyspark.sql.functions as F

conf = SparkConf().setAppName("1045").setMaster("local[4]")
spark = SparkSession.builder.config(conf = conf).getOrCreate()
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/08/01 01:16:49 WARN Utils: Your hostname, de24, resolves to a loopback address: 127.0.1.1; using 192.168.0.102 instead (on interface enp0s3)
25/08/01 01:16:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/01 01:16:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
'''
Table: Customer

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| customer_id | int     |
| product_key | int     |
+-------------+---------+
This table may contain duplicates rows. 
customer_id is not NULL.
product_key is a foreign key (reference column) to Product table.
 

Table: Product

+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| product_key | int     |
+-------------+---------+
product_key is the primary key (column with unique values) for this table.
 

Write a solution to report the customer ids from the Customer table that bought all the products in the Product table.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Customer table:
+-------------+-------------+
| customer_id | product_key |
+-------------+-------------+
| 1           | 5           |
| 2           | 6           |
| 3           | 5           |
| 3           | 6           |
| 1           | 6           |
+-------------+-------------+
Product table:
+-------------+
| product_key |
+-------------+
| 5           |
| 6           |
+-------------+
Output: 
+-------------+
| customer_id |
+-------------+
| 1           |
| 3           |
+-------------+
Explanation: 
The customers who bought all the products (5 and 6) are customers with IDs 1 and 3.
'''

In [12]:
customer_data = [
(1,5),
(2,6),
(3,5),
(3,6),
(1,6)   
]
customer_schema = ['customer_id','product_key']

product_data = [
    (5,),
    (6,)
]
product_schema = ['product_key']

In [13]:
customer_df = spark.createDataFrame(data = customer_data, schema = customer_schema)
product_df = spark.createDataFrame(data = product_data, schema = product_schema)

In [14]:
product_df.count()

2

In [16]:
customer_df.groupBy(F.col("customer_id"))\
           .agg(F.count_distinct(F.col("product_key")).alias("distinct_count"))\
           .where(F.col("distinct_count") == product_df.count())\
           .select(F.col("customer_id"))\
           .show()

+-----------+
|customer_id|
+-----------+
|          1|
|          3|
+-----------+



## SQL Solution 

<pre>
    SELECT customer_id
    FROM Customer
    GROUP BY customer_id 
    HAVING COUNT(DISTINCT product_key) = (SELECT count(*) FROM Product )
</pre>